# NEUROLOG Ablation — PyTorch Edition (GPU-Accelerated)

**Same science as `neurolog_colab.ipynb` but uses PyTorch for the neural components.**  
The original notebook ran a pure-NumPy CNN on CPU — this replaces it with a PyTorch CNN that runs on Colab's T4 GPU (10–50× faster).

The symbolic KB (clingo ASP) is unchanged.

| Strategy | Description |
|---|---|
| **Pure Neural** | Standard cross-entropy, no symbolic feedback |
| **NGA** | Neural-Guided Abduction — best single correction path |
| **WMC** | Weighted Model Counting — full probabilistic semantic loss |

**Before running**: Runtime → Change runtime type → **T4 GPU**

## 1. Setup

In [ ]:
import os

REPO_URL = 'https://github.com/konstantinoslamp/NeuroSymbolic-Network-for-Handwritten-Symbol-Understanding.git'
REPO_DIR = '/content/neurosymbolic_mvp'

if os.path.exists(REPO_DIR):
    print('Repo exists, pulling latest...')
    os.chdir(REPO_DIR)
    !git pull
else:
    !git clone {REPO_URL} {REPO_DIR}
    os.chdir(REPO_DIR)

print(f'Working directory: {os.getcwd()}')

In [ ]:
!pip install -q clingo Pillow tqdm
import sys
sys.path.insert(0, REPO_DIR)

import clingo
import torch
print(f'clingo: {clingo.__version__}')
print(f'torch:  {torch.__version__}')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
else:
    print('WARNING: No GPU found. Runtime → Change runtime type → T4 GPU')

## 2. Imports & Symbolic KB

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import time, json, copy
import matplotlib.pyplot as plt

from src.symbolic.knowledge_base import KnowledgeBase
from src.data.expression_dataset import ExpressionDataset

# Verify symbolic KB
kb = KnowledgeBase()
print('KB smoke test:')
print('  3 + 5 =', kb.deduce(3, '+', 5))
print('  7 - 2 =', kb.deduce(7, '-', 2))
print('  4 x 3 =', kb.deduce(4, '×', 3))
print('  abduce(6):', kb.abduce(6.0)[:4], '...')
print('Symbolic KB OK.')

## 3. Pre-Cache Abduction Results

All possible arithmetic results are bounded (−9 to 81 for integer ops). We pre-cache them so we never call clingo in a tight training loop.

In [ ]:
print('Pre-caching abduction results for all possible targets...')
ABDUCE_CACHE = {}

# Integer operator results: + gives 0..18, - gives -9..9, × gives 0..81
for r in range(-9, 82):
    solutions = kb.abduce(float(r))
    if solutions:
        ABDUCE_CACHE[r] = solutions
        ABDUCE_CACHE[float(r)] = solutions  # also store as float key

# Division results (rational)
for d1 in range(0, 10):
    for d2 in range(1, 10):
        r = d1 / d2
        if r not in ABDUCE_CACHE:
            solutions = kb.abduce(r)
            if solutions:
                ABDUCE_CACHE[r] = solutions

print(f'Cached {len(ABDUCE_CACHE)} result values (covers +, -, ×, ÷).')

def lookup_abduce(result):
    """Fast lookup of pre-cached abduction results."""
    if result is None:
        return []
    # Try int key first (most common)
    if result == int(result):
        sols = ABDUCE_CACHE.get(int(result))
        if sols is not None:
            return sols
    # Float key
    return ABDUCE_CACHE.get(float(result), [])

## 4. PyTorch CNN (same architecture as existing NumPy model)

In [ ]:
class SymbolCNN(nn.Module):
    """
    14-class CNN matching the existing architecture:
      Conv(1→8, 3x3) → ReLU → MaxPool(2x2) → Flatten
      → Dense(1352→128) → ReLU → Dense(128→14)

    Classes: digits 0-9 (idx 0-9), operators +,-,×,÷ (idx 10-13)
    """
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 8, kernel_size=3)   # 28→26
        self.pool  = nn.MaxPool2d(2, 2)                # 26→13
        self.fc1   = nn.Linear(8 * 13 * 13, 128)
        self.fc2   = nn.Linear(128, 14)

    def forward(self, x):
        """x: (N, 1, 28, 28) → logits (N, 14)"""
        x = F.relu(self.conv1(x))
        x = self.pool(x)
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        return self.fc2(x)

# Quick sanity check
_m = SymbolCNN().to(device)
_x = torch.randn(4, 1, 28, 28).to(device)
assert _m(_x).shape == (4, 14), 'CNN output shape wrong'
print(f'SymbolCNN OK — {sum(p.numel() for p in _m.parameters()):,} parameters')
del _m, _x

## 5. Load & Preprocess Dataset

In [ ]:
# ── Hyperparameters ──────────────────────────────────────────────────────────
TRAIN_SAMPLES = 1000   # increase to 3000 for more stable results if time allows
TEST_SAMPLES  = 400
EPOCHS        = 8
BATCH_SIZE    = 64
LR            = 1e-3
# ─────────────────────────────────────────────────────────────────────────────

OP_TO_IDX = {'+': 0, '-': 1, '×': 2, '÷': 3}
IDX_TO_OP = {v: k for k, v in OP_TO_IDX.items()}
SYM_TO_IDX = {str(i): i for i in range(10)}
SYM_TO_IDX.update({'+': 10, '-': 11, '×': 12, '÷': 13})
print('Hyperparameters set.')

In [ ]:
print('Loading datasets (downloads MNIST on first run)...')
train_ds = ExpressionDataset(num_samples=TRAIN_SAMPLES, split='train', invalid_ratio=0.05)
test_ds  = ExpressionDataset(num_samples=TEST_SAMPLES,  split='test',  invalid_ratio=0.05)
print(f'Train: {len(train_ds)} | Test: {len(test_ds)}')

def preprocess(ds):
    """Extract images (N,3,28,28), labels (d1, op, d2), results from dataset."""
    imgs, d1s, ops, d2s, ress = [], [], [], [], []
    for item in ds:
        imgs.append(item['images'])  # (3, 28, 28)
        text = item.get('text', '')
        if isinstance(text, str) and len(text) >= 3:
            syms = [text[0], text[1], text[2]]
        elif isinstance(text, (list, tuple)) and len(text) >= 3:
            syms = [str(s) for s in text[:3]]
        else:
            syms = ['0', '+', '0']
        d1s.append(SYM_TO_IDX.get(syms[0], 0))
        ops.append(SYM_TO_IDX.get(syms[1], 10))
        d2s.append(SYM_TO_IDX.get(syms[2], 0))
        ress.append(item.get('result'))
    return (
        np.array(imgs, dtype=np.float32),
        np.array(d1s, dtype=np.int64),
        np.array(ops, dtype=np.int64),
        np.array(d2s, dtype=np.int64),
        ress
    )

train_imgs, train_d1, train_op, train_d2, train_res = preprocess(train_ds)
test_imgs,  test_d1,  test_op,  test_d2,  test_res  = preprocess(test_ds)

print(f'train_imgs: {train_imgs.shape}, test_imgs: {test_imgs.shape}')
print('Data ready.')

## 6. Symbolic Loss Helpers (WMC & NGA)

In [ ]:
def softmax_np(logits):
    """Numerically stable softmax. logits: (..., K)"""
    shifted = logits - logits.max(axis=-1, keepdims=True)
    e = np.exp(shifted)
    return e / (e.sum(axis=-1, keepdims=True) + 1e-12)


def wmc_loss_and_grad(logits_np, results):
    """
    Compute WMC semantic loss and its gradient w.r.t. logits.

    logits_np : (B, 3, 14)  — one row per expression, 3 positions
    results   : list of B values (float or None for invalid expressions)

    Returns
    -------
    loss : float scalar
    grad : (B, 3, 14) gradient of loss w.r.t. logits_np
    """
    probs = softmax_np(logits_np)          # (B, 3, 14)
    B = logits_np.shape[0]
    grad = np.zeros_like(logits_np)
    total_loss, valid_n = 0.0, 0

    for i in range(B):
        r = results[i]
        if r is None:
            continue
        solutions = lookup_abduce(r)
        if not solutions:
            continue

        p_d1 = probs[i, 0, :10]          # P(digit | pos 0)
        p_op = probs[i, 1, 10:14]        # P(op    | pos 1)
        p_d2 = probs[i, 2, :10]          # P(digit | pos 2)

        exp_probs = np.array([
            p_d1[d1] * p_op[OP_TO_IDX[op]] * p_d2[d2]
            for (d1, op, d2) in solutions
        ])
        wmc = exp_probs.sum()
        if wmc < 1e-12:
            continue

        total_loss += -np.log(wmc + 1e-12)
        valid_n += 1

        # Compute WMC target distribution (normalised explanation weights)
        weights = exp_probs / wmc         # (K,)
        tgt_d1 = np.zeros(14)
        tgt_op = np.zeros(14)
        tgt_d2 = np.zeros(14)
        for w, (d1, op, d2) in zip(weights, solutions):
            tgt_d1[d1]             += w
            tgt_op[10 + OP_TO_IDX[op]] += w
            tgt_d2[d2]             += w

        # Gradient: prob - target  (cross-entropy over valid explanations)
        grad[i, 0] = probs[i, 0] - tgt_d1
        grad[i, 1] = probs[i, 1] - tgt_op
        grad[i, 2] = probs[i, 2] - tgt_d2

    denom = max(valid_n, 1)
    return total_loss / denom, grad / denom


def nga_targets(logits_np, results):
    """
    NGA: for each sample, find the highest-probability valid explanation
    given the true result and return it as the training target.
    If the current prediction is already correct, reinforce it.

    Returns (tgt_d1, tgt_op_global, tgt_d2) as int arrays (global class idx).
    """
    probs  = softmax_np(logits_np)         # (B, 3, 14)
    B      = logits_np.shape[0]

    pred_d1 = np.argmax(probs[:, 0, :10],   axis=1)   # 0-9
    pred_op = np.argmax(probs[:, 1, 10:14], axis=1)   # 0-3
    pred_d2 = np.argmax(probs[:, 2, :10],   axis=1)   # 0-9

    tgt_d1 = pred_d1.copy()
    tgt_op = pred_op.copy() + 10    # global idx 10-13
    tgt_d2 = pred_d2.copy()

    for i in range(B):
        r = results[i]
        if r is None:
            continue

        # Check if current prediction is already correct
        op_sym = IDX_TO_OP[pred_op[i]]
        try:
            if op_sym == '÷':
                dv = int(pred_d2[i])
                computed = float(pred_d1[i]) / dv if dv != 0 else None
            else:
                computed = kb.deduce(int(pred_d1[i]), op_sym, int(pred_d2[i]))
            if computed is not None and abs(float(computed) - float(r)) < 1e-6:
                continue   # correct — reinforce current prediction
        except Exception:
            pass

        # Abduce best explanation
        solutions = lookup_abduce(r)
        if not solutions:
            continue

        best_p, best_expr = -1.0, None
        for (d1, op, d2) in solutions:
            oi = OP_TO_IDX[op]
            p = probs[i, 0, d1] * probs[i, 1, 10 + oi] * probs[i, 2, d2]
            if p > best_p:
                best_p, best_expr = p, (d1, op, d2)

        if best_expr:
            d1, op, d2 = best_expr
            tgt_d1[i] = d1
            tgt_op[i] = 10 + OP_TO_IDX[op]
            tgt_d2[i] = d2

    return tgt_d1, tgt_op, tgt_d2


print('Symbolic helpers defined.')

## 7. Training Loop

In [ ]:
def run_training(strategy, initial_state_dict=None):
    """
    Train SymbolCNN with one of three strategies.
    strategy: 'pure_neural' | 'nga' | 'wmc'
    initial_state_dict: if provided, all runs start from identical weights.
    Returns (model, history, elapsed_seconds)
    """
    assert strategy in ('pure_neural', 'nga', 'wmc'), f'Unknown strategy: {strategy}'

    model = SymbolCNN().to(device)
    if initial_state_dict is not None:
        model.load_state_dict(copy.deepcopy(initial_state_dict))

    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    ce_loss_fn = nn.CrossEntropyLoss()

    N = train_imgs.shape[0]
    history = []
    t0 = time.time()

    label_names = {'pure_neural': 'Pure Neural', 'nga': 'NGA', 'wmc': 'WMC'}
    print(f"\n{'='*55}")
    print(f"  {label_names[strategy]} — {EPOCHS} epochs × {N} samples")
    print(f"{'='*55}")

    for epoch in range(EPOCHS):
        model.train()
        perm = np.random.permutation(N)
        epoch_loss, epoch_correct, n_batches = 0.0, 0, 0

        for start in range(0, N, BATCH_SIZE):
            idx  = perm[start:start + BATCH_SIZE]
            B    = len(idx)

            # imgs_b: (B, 3, 28, 28) → imgs_flat: (B*3, 1, 28, 28)
            # Layout: [s0_pos0, s0_pos1, s0_pos2, s1_pos0, ...]
            imgs_b    = torch.FloatTensor(train_imgs[idx])
            imgs_flat = imgs_b.view(B * 3, 1, 28, 28).to(device)

            d1_b  = train_d1[idx]       # (B,) global idx 0-9
            op_b  = train_op[idx]       # (B,) global idx 10-13
            d2_b  = train_d2[idx]       # (B,) global idx 0-9
            res_b = [train_res[i] for i in idx]

            # Ground-truth labels, interleaved to match imgs_flat order
            gt_labels = np.empty(B * 3, dtype=np.int64)
            gt_labels[0::3] = d1_b
            gt_labels[1::3] = op_b
            gt_labels[2::3] = d2_b

            # ── Pure Neural ──────────────────────────────────────────
            if strategy == 'pure_neural':
                optimizer.zero_grad()
                logits_flat = model(imgs_flat)             # (B*3, 14)
                labels_t    = torch.LongTensor(gt_labels).to(device)
                loss        = ce_loss_fn(logits_flat, labels_t)
                loss.backward()
                optimizer.step()

                epoch_loss    += loss.item()
                preds          = logits_flat.argmax(dim=1).cpu().numpy()
                epoch_correct += int(np.sum(preds == gt_labels))

            # ── NGA ──────────────────────────────────────────────────
            elif strategy == 'nga':
                optimizer.zero_grad()
                logits_flat = model(imgs_flat)             # (B*3, 14)
                logits_3d   = logits_flat.detach().cpu().numpy().reshape(B, 3, 14)

                tgt_d1, tgt_op, tgt_d2 = nga_targets(logits_3d, res_b)

                nga_lbls = np.empty(B * 3, dtype=np.int64)
                nga_lbls[0::3] = tgt_d1
                nga_lbls[1::3] = tgt_op
                nga_lbls[2::3] = tgt_d2

                labels_t = torch.LongTensor(nga_lbls).to(device)
                loss     = ce_loss_fn(logits_flat, labels_t)
                loss.backward()
                optimizer.step()

                epoch_loss    += loss.item()
                preds          = logits_flat.argmax(dim=1).cpu().numpy()
                epoch_correct += int(np.sum(preds == nga_lbls))

            # ── WMC ──────────────────────────────────────────────────
            elif strategy == 'wmc':
                optimizer.zero_grad()
                logits_flat  = model(imgs_flat)            # (B*3, 14)
                logits_3d    = logits_flat.detach().cpu().numpy().reshape(B, 3, 14)

                loss_val, grad_3d = wmc_loss_and_grad(logits_3d, res_b)

                # Inject WMC gradient back through PyTorch graph
                grad_flat = torch.FloatTensor(grad_3d.reshape(B * 3, 14)).to(device)
                logits_flat.backward(grad_flat)
                optimizer.step()

                epoch_loss    += loss_val
                preds          = logits_3d.reshape(B*3, 14).argmax(axis=1)
                epoch_correct += int(np.sum(preds == gt_labels))

            n_batches += 1

        avg_loss = epoch_loss   / max(n_batches, 1)
        avg_acc  = epoch_correct / (N * 3)
        history.append({'epoch': epoch + 1, 'avg_loss': avg_loss, 'accuracy': avg_acc})
        print(f'  Epoch {epoch+1:2d}/{EPOCHS}  loss={avg_loss:.4f}  acc={avg_acc:.4f}')

    elapsed = time.time() - t0
    print(f'  Done in {elapsed:.1f}s')
    return model, history, elapsed


print('Training function defined.')

## 8. Evaluation Helper

In [ ]:
def evaluate_model(model):
    """
    Evaluate model on the test set.
    Returns dict: digit_acc, op_acc, expr_acc, result_acc, ece
    """
    model.eval()
    N = test_imgs.shape[0]

    all_preds_d1, all_preds_op, all_preds_d2 = [], [], []
    all_conf_d1, all_conf_op, all_conf_d2     = [], [], []

    with torch.no_grad():
        for start in range(0, N, BATCH_SIZE):
            end   = min(start + BATCH_SIZE, N)
            B     = end - start
            imgs_b    = torch.FloatTensor(test_imgs[start:end])
            imgs_flat = imgs_b.view(B * 3, 1, 28, 28).to(device)
            logits_flat = model(imgs_flat).cpu().numpy()    # (B*3, 14)
            probs_flat  = softmax_np(logits_flat)

            # Positions: [0::3]=d1, [1::3]=op, [2::3]=d2
            pr_d1 = probs_flat[0::3]   # (B, 14)
            pr_op = probs_flat[1::3]
            pr_d2 = probs_flat[2::3]

            all_preds_d1.extend(np.argmax(pr_d1[:, :10],   axis=1))
            all_preds_op.extend(np.argmax(pr_op[:, 10:14], axis=1))   # 0-3
            all_preds_d2.extend(np.argmax(pr_d2[:, :10],   axis=1))

            all_conf_d1.extend(pr_d1[:, :10].max(axis=1))
            all_conf_op.extend(pr_op[:, 10:14].max(axis=1))
            all_conf_d2.extend(pr_d2[:, :10].max(axis=1))

    pred_d1 = np.array(all_preds_d1)   # 0-9
    pred_op = np.array(all_preds_op)   # 0-3
    pred_d2 = np.array(all_preds_d2)   # 0-9

    true_d1 = test_d1                  # 0-9
    true_op = test_op - 10             # 0-3
    true_d2 = test_d2                  # 0-9

    # Per-symbol accuracies
    digit_acc = (np.mean(pred_d1 == true_d1) + np.mean(pred_d2 == true_d2)) / 2.0
    op_acc    = float(np.mean(pred_op == true_op))

    # Expression accuracy: all 3 positions correct
    expr_acc  = float(np.mean(
        (pred_d1 == true_d1) & (pred_op == true_op) & (pred_d2 == true_d2)
    ))

    # Result accuracy: predicted expression evaluates to correct result
    r_correct, r_total = 0, 0
    for i in range(N):
        r = test_res[i]
        if r is None:
            continue
        r_total += 1
        op_sym = IDX_TO_OP[pred_op[i]]
        try:
            if op_sym == '÷':
                dv = int(pred_d2[i])
                computed = float(pred_d1[i]) / dv if dv != 0 else None
            else:
                computed = kb.deduce(int(pred_d1[i]), op_sym, int(pred_d2[i]))
            if computed is not None and abs(float(computed) - float(r)) < 1e-6:
                r_correct += 1
        except Exception:
            pass

    result_acc = r_correct / max(r_total, 1)

    # ECE (Expected Calibration Error) — 10-bin calibration
    confs   = np.concatenate([all_conf_d1, all_conf_d2, all_conf_op])
    corrects_d1 = (pred_d1 == true_d1).astype(float)
    corrects_d2 = (pred_d2 == true_d2).astype(float)
    corrects_op = (pred_op == true_op).astype(float)
    corrects = np.concatenate([corrects_d1, corrects_d2, corrects_op])

    ece = 0.0
    n_bins = 10
    bin_edges = np.linspace(0, 1, n_bins + 1)
    for b in range(n_bins):
        in_bin = (confs >= bin_edges[b]) & (confs < bin_edges[b + 1])
        if in_bin.sum() == 0:
            continue
        avg_conf = confs[in_bin].mean()
        avg_acc  = corrects[in_bin].mean()
        ece += (in_bin.sum() / len(confs)) * abs(avg_conf - avg_acc)

    return {
        'digit_acc':  float(digit_acc),
        'op_acc':     float(op_acc),
        'expr_acc':   float(expr_acc),
        'result_acc': float(result_acc),
        'ece':        float(ece),
    }


print('Evaluation function defined.')

## 9. Run All Ablations

All three strategies start from the **same initial weights** for a fair comparison.

In [ ]:
# Shared initial weights — all runs start from the same random initialisation
reference_model = SymbolCNN().to(device)
initial_state = copy.deepcopy(reference_model.state_dict())

strategies  = ['pure_neural', 'nga', 'wmc']
label_names = {'pure_neural': 'Pure Neural', 'nga': 'NGA', 'wmc': 'WMC'}

all_results = {}
wall_t0 = time.time()

for strat in strategies:
    model_s, history_s, elapsed_s = run_training(strat, initial_state_dict=initial_state)
    eval_s = evaluate_model(model_s)
    all_results[strat] = {
        'label':   label_names[strat],
        'history': history_s,
        'eval':    eval_s,
        'time':    elapsed_s,
    }
    print(f'  → Digit Acc={eval_s["digit_acc"]:.3f}  Expr Acc={eval_s["expr_acc"]:.3f}  '
          f'Result Acc={eval_s["result_acc"]:.3f}  ECE={eval_s["ece"]:.4f}')

print(f'\nTotal wall time: {time.time() - wall_t0:.1f}s')

## 10. Results Table (Paper-Ready)

In [ ]:
print('\n' + '='*80)
print('  ABLATION STUDY RESULTS  (MATH-3 dataset, PyTorch CNN, T4 GPU)')
print('='*80)
hdr = f'{"Strategy":<25} {"Digit":>8} {"Op":>8} {"Expr":>8} {"Result":>8} {"ECE":>8} {"Time(s)":>8}'
print(hdr)
print('-'*80)
for strat in strategies:
    r = all_results[strat]
    ev = r['eval']
    print(f"{r['label']:<25} "
          f"{ev['digit_acc']:>8.3f} "
          f"{ev['op_acc']:>8.3f} "
          f"{ev['expr_acc']:>8.3f} "
          f"{ev['result_acc']:>8.3f} "
          f"{ev['ece']:>8.4f} "
          f"{r['time']:>8.1f}")
print('='*80)

print('\nLaTeX snippet:')
print(r'\begin{table}[h]\centering')
print(r'\begin{tabular}{lccccc}')
print(r'\toprule')
print(r'Strategy & Digit Acc & Op Acc & Expr Acc & Result Acc & ECE \\')
print(r'\midrule')
for strat in strategies:
    r = all_results[strat]
    ev = r['eval']
    print(f"{r['label']} & {ev['digit_acc']:.3f} & {ev['op_acc']:.3f} & "
          f"{ev['expr_acc']:.3f} & {ev['result_acc']:.3f} & {ev['ece']:.4f} \\\\")
print(r'\bottomrule')
print(r'\end{tabular}')
print(r'\caption{Ablation: Pure Neural vs NGA vs WMC on MATH(3).}')
print(r'\label{tab:ablation}')
print(r'\end{table}')

## 11. Training Curves

In [ ]:
colors = {'pure_neural': '#e74c3c', 'nga': '#3498db', 'wmc': '#2ecc71'}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for strat in strategies:
    r   = all_results[strat]
    h   = r['history']
    eps = [e['epoch']    for e in h]
    ls  = [e['avg_loss'] for e in h]
    acs = [e['accuracy'] for e in h]
    lbl = r['label']
    c   = colors[strat]
    axes[0].plot(eps, ls,  color=c, label=lbl, linewidth=2, marker='o', markersize=4)
    axes[1].plot(eps, acs, color=c, label=lbl, linewidth=2, marker='o', markersize=4)

for ax, title, ylabel in zip(axes,
                              ['Training Loss', 'Training Accuracy'],
                              ['Loss', 'Accuracy']):
    ax.set_xlabel('Epoch'); ax.set_ylabel(ylabel)
    ax.set_title(title); ax.legend(); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('ablation_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: ablation_curves.png')

In [ ]:
metrics   = ['Digit Acc', 'Op Acc', 'Expr Acc', 'Result Acc']
keys      = ['digit_acc', 'op_acc', 'expr_acc', 'result_acc']
x         = np.arange(len(metrics))
width     = 0.25

fig, ax = plt.subplots(figsize=(12, 6))
for i, strat in enumerate(strategies):
    vals = [all_results[strat]['eval'][k] for k in keys]
    bars = ax.bar(x + i * width, vals, width,
                  label=all_results[strat]['label'],
                  color=colors[strat], alpha=0.85)
    ax.bar_label(bars, fmt='%.2f', fontsize=8, padding=2)

ax.set_ylabel('Accuracy'); ax.set_ylim(0, 1.1)
ax.set_title('Ablation Study: Pure Neural vs NGA vs WMC')
ax.set_xticks(x + width); ax.set_xticklabels(metrics)
ax.legend(); ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('ablation_barplot.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: ablation_barplot.png')

## 12. Save & Download Results

In [ ]:
os.makedirs('results', exist_ok=True)
output_path = 'results/neurolog_torch_ablation.json'

def to_python(obj):
    if isinstance(obj, (np.integer,)):  return int(obj)
    if isinstance(obj, (np.floating,)): return float(obj)
    if isinstance(obj, np.ndarray):     return obj.tolist()
    if isinstance(obj, dict):           return {k: to_python(v) for k, v in obj.items()}
    if isinstance(obj, list):           return [to_python(v) for v in obj]
    return obj

with open(output_path, 'w') as f:
    json.dump(to_python(all_results), f, indent=2)
print(f'Saved {os.path.getsize(output_path)/1024:.1f} KB → {output_path}')

try:
    from google.colab import files
    files.download(output_path)
    files.download('ablation_curves.png')
    files.download('ablation_barplot.png')
    print('Downloads triggered.')
except ImportError:
    print('Not in Colab — files saved locally.')